# Controlling MODFLOW 6 with the API — B. Monitoring convergence

This is part B of the MODFLOW 6 API series — see [**A. Basic usage**](mf6-api-a.ipynb) for the API lifecycle (`initialize → update`/`solve → finalize`) and the simulation → models → packages mental model.

Step into the solver and watch how the solution converges: after preparing each time step, loop over the solver's outer iterations by hand, read the maximum head change at every iteration, and refresh a live plot after each time step so you can watch convergence happen as the model runs.

Run the imports and library-location cells below first, then continue to the convergence monitor.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline

import pathlib as pl

import flopy
import matplotlib.pyplot as plt
from mf6_notebook_helpers import find_mf6_libraries, plot_convergence
from modflowapi import ModflowApi

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. The `find_mf6_libraries` helper in `mf6_notebook_helpers.py` locates the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the `mf6` executable inside the active pixi environment; confirm that both exist.

In [ ]:
# libmf6 (the shared library the API drives) and the mf6 executable,
# located in the active pixi environment
lib_name, mf6_exe = find_mf6_libraries()

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## Monitor convergence

Step into the solver to watch how the solution converges. After preparing each time step, loop over the solver's outer iterations manually and read the maximum head change (`HNCG`) at every iteration. The `|maximum head change|` for every iteration is plotted on a log scale and the figure is refreshed after each time step, so you can watch convergence happen **as the model runs**. The x-axis is grouped by stress period and time step (read from the `TDIS` package, with labels thinned when there are many); each time step is given an **equal-width slot** so the lengthy initial steady-state solve leaves enough room for the equally interesting transient time steps. The figure uses the USGS publication style from `flopy.plot.styles`. This exposes the inner workings that are normally hidden when MODFLOW 6 runs on its own, and is useful for diagnosing slow or failing convergence.

In [ ]:
ws = pl.Path(f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}")
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api_b")

sim = flopy.mf6.MFSimulation.load(
    sim_name=name,
    sim_ws=ws,
    write_headers=False,
)
sim.memory_print_option = "all"
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
mf6 = ModflowApi(
    lib_name,
    working_directory=new_ws,
)
mf6.initialize()

# solver settings and the live pointer to the maximum head change per outer
# iteration (HNCG); KPER and KSTP are read from the TDIS package each time step
max_iter = int(mf6.get_value(mf6.get_var_address("MXITER", "SLN_1"))[0])
max_change = mf6.get_value_ptr(mf6.get_var_address("HNCG", "SLN_1"))
kper_tag = mf6.get_var_address("KPER", "TDIS")
kstp_tag = mf6.get_var_address("KSTP", "TDIS")

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

# convergence history, one entry per outer iteration:
# (cumulative iteration, |maximum head change|, stress period, time step).
# plot_convergence() (in mf6_notebook_helpers) turns it into the live plot.
history = []
not_converged = []

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)

    niter = 0
    while current_time < end_time:
        dt = mf6.get_time_step()
        mf6.prepare_time_step(dt)
        kper = int(mf6.get_value(kper_tag)[0])
        kstp = int(mf6.get_value(kstp_tag)[0])

        mf6.prepare_solve()
        kiter = 0
        has_converged = False
        while kiter < max_iter:
            has_converged = mf6.solve()
            niter += 1
            history.append((niter, abs(float(max_change[kiter])), kper, kstp))
            kiter += 1
            if has_converged:
                break

        plot_convergence(fig, ax, history)  # live update once per time step

        if not has_converged:
            not_converged.append((kper, kstp))

        mf6.finalize_solve()
        mf6.finalize_time_step()
        current_time = mf6.get_current_time()

    mf6.finalize()

plt.close(fig)  # the live figure is already displayed; avoid a duplicate

if not_converged:
    print("did not converge:", ", ".join(f"SP {p} TS {t}" for p, t in not_converged))
else:
    print("all time steps converged")

**What to look for.** Each marker is one outer (Picard) iteration. Within a time step the curve drops as the head change shrinks toward convergence, then resets at the start of the next time step — the vertical lines and `SP`/`TS` labels mark those breaks. A time step that reaches a small head change in just a few iterations converged easily; a curve that stays high or runs all the way to the `MXITER` cap flags a time step that is hard to solve. This is exactly the diagnostic information MODFLOW normally hides when it runs on its own.

## Recap

In this notebook you:
- Opened the solver's outer-iteration loop by hand and read the maximum head change (`HNCG`) at each iteration.
- Refreshed a convergence plot after every time step to watch the solution converge as the model ran.

This exposes the diagnostic information MODFLOW 6 normally hides when it runs on its own, which is useful for finding slow or failing convergence.